# AGAR-RL V11: fixed-seed checkpoint evaluation

This notebook evaluates every saved V11 checkpoint, including the latest alias, on the same held-out seeds and the same V11 environment. It reports peak mass, final mass, survival to the time limit, simulated seconds, reward, pellets, and eliminations. Partial results are written after each checkpoint, so the evaluation can be interrupted and resumed by rerunning it.

The replay uses the latest V11 policy. All inputs and outputs stay in the V11 Drive folder; V10 and older model folders are not searched.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess, sys
REPO='/content/agario'
if not os.path.isdir(os.path.join(REPO,'.git')):
    subprocess.run(['git','clone','https://github.com/Albin0903/agario.git',REPO],check=True)
os.chdir(REPO)
subprocess.run(['git','fetch','origin','main'],check=True)
subprocess.run(['git','checkout','-B','main','origin/main'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'],check=True)
print('V11 evaluation code is ready.')


## Compare the saved checkpoints
The default evaluates all immutable `ppo_step_*.zip` checkpoints. Each checkpoint is paired with its same-step VecNormalize snapshot; the rolling `ppo_latest.zip` alias is deduplicated. `EPISODES_PER_CHECKPOINT` can be raised for tighter confidence or `CHECKPOINT_LIMIT` can select an evenly spaced subset for a quick preview. Every model sees the same seed list.


In [ ]:
import json, subprocess, sys
from pathlib import Path
import pandas as pd

DRIVE_V11=Path('/content/drive/MyDrive/agario_rl_backup_v11')
EVAL_DIR=DRIVE_V11/'evaluation'
EVAL_DIR.mkdir(parents=True,exist_ok=True)
if not (DRIVE_V11/'v11_manifest.json').is_file():
    raise FileNotFoundError('No V11 manifest found. Run notebooks/train_colab.ipynb first.')
EPISODES_PER_CHECKPOINT=5
CHECKPOINT_LIMIT=0  # 0 means every checkpoint
cmd=[sys.executable,'-m','src.analysis.evaluate_v11',
     '--checkpoint-dir',str(DRIVE_V11),
     '--env-config','config/env_config.yaml',
     '--episodes',str(EPISODES_PER_CHECKPOINT),
     '--seed','10000',
     '--limit-checkpoints',str(CHECKPOINT_LIMIT),
     '--output-csv',str(EVAL_DIR/'v11_evaluation.csv'),
     '--output-json',str(EVAL_DIR/'v11_evaluation.json')]
subprocess.run(cmd,check=True)
results=pd.read_csv(EVAL_DIR/'v11_evaluation.csv')
summary=(results.groupby(['checkpoint_step','checkpoint'])
         .agg(peak_mass_mean=('peak_mass','mean'), peak_mass_std=('peak_mass','std'),
              final_mass_mean=('final_mass','mean'), survival_rate=('survived_to_time_limit','mean'),
              simulated_seconds_mean=('simulated_seconds','mean'), pellets_mean=('pellets_eaten','mean'))
         .reset_index().sort_values('checkpoint_step'))
display(summary.tail(20))


In [ ]:
import matplotlib.pyplot as plt
fig,axes=plt.subplots(3,1,figsize=(12,12),sharex=True)
axes[0].plot(summary['checkpoint_step'],summary['peak_mass_mean'],marker='o',label='mean peak mass')
axes[0].fill_between(summary['checkpoint_step'],
                     summary['peak_mass_mean']-summary['peak_mass_std'].fillna(0),
                     summary['peak_mass_mean']+summary['peak_mass_std'].fillna(0),alpha=.2)
axes[0].set_ylabel('Peak mass')
axes[1].plot(summary['checkpoint_step'],summary['final_mass_mean'],marker='o',color='tab:orange')
axes[1].set_ylabel('Final mass')
axes[2].plot(summary['checkpoint_step'],summary['survival_rate'],marker='o',color='tab:green')
axes[2].set_ylabel('Survival fraction')
axes[2].set_xlabel('V11 training timestep')
for ax in axes: ax.grid(alpha=.25)
fig.suptitle('V11 fixed-seed checkpoint comparison')
fig.tight_layout()
fig.savefig(EVAL_DIR/'v11_checkpoint_comparison.png',dpi=160,bbox_inches='tight')
display(fig)
ranked=summary.sort_values(['peak_mass_mean','survival_rate','final_mass_mean'],ascending=False)
print('Best by peak mass, then survival, then final mass:')
display(ranked.head(10))


## Training-time scenario trend (every 1M steps)

The trainer evaluates five fixed seeds in each controlled scenario every million steps. These are simulation stress tests (standard, half-food, and 150% bot count), not official Agar.io modes. Split credit means a split decision was followed by a kill within 30 agent decisions; it is a diagnostic attribution, not a causal guarantee.


In [ ]:
scenario_file=DRIVE_V11/'scenario_evaluations.jsonl'
if scenario_file.is_file():
    scenario_records=[]
    for line in scenario_file.read_text(encoding='utf-8').splitlines():
        try: scenario_records.append(json.loads(line))
        except json.JSONDecodeError: continue  # ignore a final partial row after an interrupted write
    scenario_rows=pd.DataFrame(scenario_records)
    if scenario_rows.empty:
        print('No complete scenario rows found yet.')
    else:
        scenario_summary=(scenario_rows.groupby(['evaluation_milestone','scenario'])
            .agg(peak_mass_mean=('peak_mass','mean'), final_mass_mean=('final_mass','mean'),
                 survival_rate=('survived_to_time_limit','mean'), split_kill_rate=('split_kill_rate_30','mean'),
                 kills_mean=('kills','mean'), split_actions_mean=('split_actions','mean'))
            .reset_index().sort_values(['evaluation_milestone','scenario']))
        display(scenario_summary)
        fig,axes=plt.subplots(3,1,figsize=(12,11),sharex=True)
        for scenario,group in scenario_summary.groupby('scenario'):
            x=group['evaluation_milestone']/1_000_000
            axes[0].plot(x,group['peak_mass_mean'],marker='o',label=scenario)
            axes[1].plot(x,group['survival_rate'],marker='o',label=scenario)
            axes[2].plot(x,group['split_kill_rate'],marker='o',label=scenario)
        axes[0].set_ylabel('Peak mass (mean)')
        axes[1].set_ylabel('Survival fraction')
        axes[2].set_ylabel('Split→kill within 30 steps')
        axes[2].set_xlabel('Training milestone (millions of steps)')
        for ax in axes: ax.grid(alpha=.25); ax.legend()
        fig.tight_layout(); display(fig)
else:
    print('No per-million scenario evaluations found yet.')


## Replay the latest checkpoint
The replay subprocess disables optional TensorBoard/TensorFlow initialization and audio output. It fails loudly if the V11 model cannot load, rather than substituting another policy.

In [ ]:
latest=DRIVE_V11/'ppo_latest.zip'
if not latest.is_file(): raise FileNotFoundError(latest)
replay=EVAL_DIR/'latest_v11_replay.mp4'
subprocess.run([sys.executable,'src/inference/record_match.py',
                '--model',str(latest),'--output',str(replay),
                '--steps','2400','--fps','30','--deterministic'],check=True)
print('Replay saved to',replay)
from IPython.display import Video,display
display(Video(str(replay),embed=True))
